In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
from pathlib import Path
import json
import re
import requests
import pandas as pd
import torch

In [ ]:
print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))

True 1 NVIDIA GeForce RTX 2080 Ti


**SPACING NORMALIZATION**

In [ ]:
#normalizing spacing before sending to LLM
def normalize_spacing(text):
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    paragraphs = []
    current_paragraph = []

    for line in text.split("\n"):
        cleaned_line = " ".join(line.split())

        if cleaned_line:
            current_paragraph.append(cleaned_line)
        elif current_paragraph:
            paragraphs.append(" ".join(current_paragraph))
            current_paragraph = []

    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph))

    return "\n\n".join(paragraphs)

In [ ]:
INPUT_CSV = Path("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92.csv")

df = pd.read_csv(INPUT_CSV)
df["text"] = df["text"].apply(normalize_spacing)
df.to_csv("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92_NORM.csv", index=False, encoding="utf-8")
df.to_excel("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92_NORM.xlsx", index=False)


**LLM-ASSISTED MASKING**

In [ ]:
INPUT_CSV = Path("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92_NORM.csv")
OUTPUT_CSV = Path("SCOTBESS_DATASET_MASKED.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text"
OUTPUT_COL = "masked_text"
ENTITIES_COL = "llm_entities_json"
REJECTED_COL = "llm_rejected_entities_json"
STATUS_COL = "llm_masking_status"
ERROR_COL = "llm_masking_error"

MODEL = "qwen2.5:14b-instruct"
OLLAMA_URL = "http://127.0.0.1:11435/api/chat"
TIMEOUT = 1500
CHECKPOINT_EVERY = 10

In [ ]:
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "PERSON": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},
            
        "PLACE": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},

        "EMAIL": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},

        "PHONE": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},

        "LINK": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},

        "APPLICATION_CODE": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},

        "ORGANIZATION": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True},



    },
    "required": [
        "PERSON",
        "PLACE",
        "EMAIL",
        "PHONE",
        "LINK",
        "APPLICATION_CODE",
        "ORGANIZATION"],
    "additionalProperties": False}

In [ ]:
SYSTEM_PROMPT = """
Identify personal information and project-identifying proper names that should
be masked before multi-label topic classification.

Do not attempt to identify every named entity. Retain generic words and
substantive topic information. When uncertain, omit the value.

Analyse only the content between <BEGIN_TEXT> and <END_TEXT>.
Every returned value must occur character-for-character in the input.
Return each distinct value once and under only one category.

CATEGORIES

PERSON - the exact proper name of an identifiable individual, including a full
name, first name used as a personal name, surname, or title combined with a name.

DO NOT return occupations, unnamed people, roles, organisations, or groups of
people.

PLACE - an exact proper name or identifying expression referring to a specific
place, address, named property, road, facility, or geographical feature.

This includes:
* complete or partial residential or postal addresses;
* house numbers, postcodes, house names, and named private properties;
* named roads, streets, lanes, avenues, drives, ways, motorways, routes,
  and tracks;
* named cities, towns, villages, counties, neighbourhoods, rivers, farms,
  buildings, landmarks, facilities, and geographical features.

Return the complete identifying expression as it occurs in the text.
A phrase that merely describes a place is not PLACE.

DO NOT return generic place descriptions such as "the village", "this area", "the site", "the access road", "nearby
roads", or "the substation".

EMAIL - an email address.

PHONE - a telephone or mobile number.

LINK - a URL, web address, document link, or PDF link.

APPLICATION_CODE - only a short alphanumeric code or reference number issued
by a council or agency for a specific application, appeal, consultation, or
case.

DO NOT return dates, notice names, document titles, legislation references,
section numbers, measurements, or ordinary numbers.

ORGANIZATION - only the complete proper name of a company, developer, applicant,
contractor, consultancy, campaign group, or local association that is a named party
to the specific case.

If both an organisation's full name and its abbreviation appear in the input,
return both as separate values.

An organisation remains ORGANIZATION even when its name contains a
geographical word.

DISAMBIGUATION
* A project-specific company or group is ORGANIZATION rather than PLACE.
* Each value may appear under only one category.
* When the category or entity boundary is uncertain, omit the value.

DO NOT REPORT
These exclusions override all category definitions.

Never return:
* Never return a common noun, generic role, unnamed group, or descriptive phrase.
A value qualifies only if it contains an explicit proper identifying name.
This applies even when the surrounding context makes clear which specific person,
place, or organisation the phrase refers to. For example, never return "the council", "the applicant", "the developer",
"residents", "the substation", "the village", "this area", "residential area",
"local fire services", "the emergency services", or "the road".
* Country names.
* Technical, project-type, planning, or industry terms, such as "BESS".
* Descriptions of administrative events, notices, or procedures that do not
  contain an actual reference code.

OUTPUT
Return only valid JSON. Do not include an explanation or Markdown formatting.
Return exactly this object structure with every key present:

{
  "PERSON": [],
  "PLACE": [],
  "EMAIL": [],
  "PHONE": [],
  "LINK": [],
  "APPLICATION_CODE": [],
  "ORGANIZATION": []
}
"""

In [ ]:
#priority order for resolving cross-category overlaps when masking
#higher number = masked first 
CATEGORY_PRIORITY = {
    "EMAIL": 100,
    "PHONE": 100,
    "LINK": 100,
    "APPLICATION_CODE": 100,
    "PERSON": 95,
    "ORGANIZATION": 80,
    "PLACE": 70}

In [ ]:
def choose_num_ctx(text):
    words = len(text.split())

    if words <= 2600:
        return 8192
    elif words <= 4000:
        return 12288
    elif words <= 6200:
        return 16384
    else:
        return 18432


def choose_num_predict(text):
    words = len(text.split())

    if words <= 2600:
        return 2048
    elif words <= 6200:
        return 3072
    else:
        return 4096


def run_ollama_chat(text):
    num_ctx = choose_num_ctx(text)
    num_predict = choose_num_predict(text)

    user_content = ("Detect all identifying entities in this document and return only the fixed JSON object described in your instructions.\n\n"
    "<BEGIN_TEXT>" + text + "<END_TEXT>")

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},],
        "format": OUTPUT_SCHEMA,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0,
            "repeat_penalty": 1.0,
            "seed": 42,
            "top_k": 1,
            "num_ctx": num_ctx,
            "num_predict": num_predict},}

    response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()


def parse_categories(raw_output):
    parsed = json.loads(raw_output)

    if not isinstance(parsed, dict):
        raise ValueError("The model output is not a JSON object.")

    categories = {}
    for category in CATEGORY_PRIORITY:
        values = parsed.get(category, [])
        if not isinstance(values, list):
            raise ValueError(f'"{category}" is not a list.')
        #keep only non-empty strings, drop duplicates, preserve order
        seen = set()
        clean = []
        for v in values:
            if isinstance(v, str) and v.strip() and v not in seen:
                seen.add(v)
                clean.append(v)
        categories[category] = clean

    return categories


def split_found_and_missing(text, categories):
    """
    Keeps only strings that actually occur verbatim in the source text.
    Anything the model returned that can't be found is logged separately
    rather than silently dropped or forced into the mask.
    """
    found, missing = {}, {}

    for category, values in categories.items():
        found[category] = [v for v in values if v in text]
        missing[category] = [v for v in values if v not in text]

    return found, missing


def apply_mask(text, categories):
    masked = text
    #most specific category first, longest string first within a category, so a short match (e.g. "Fife") can't corrupt a longer one that contains it (e.g. "Fife Council") before it gets its own turn
    for category in sorted(categories, key=lambda c: -CATEGORY_PRIORITY[c]):
        values = sorted(set(categories[category]), key=len, reverse=True)

        for value in values:
            pattern = re.escape(value)
            if re.match(r"^\w", value):
                pattern = r"\b" + pattern
            if re.search(r"\w$", value):
                pattern = pattern + r"\b"
            masked = re.sub(pattern, f"[{category}]", masked)

    return masked

In [ ]:
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)
    print(f"Resuming from {OUTPUT_CSV}")
else:
    df = pd.read_csv(INPUT_CSV)
    print(f"Loaded {INPUT_CSV}")

for column in [OUTPUT_COL, ENTITIES_COL, REJECTED_COL, STATUS_COL, ERROR_COL]:
    if column not in df.columns:
        df[column] = pd.NA

num_rows = len(df)
print(f"Found {num_rows} rows.\n")

for position, (index, row) in enumerate(df.iterrows(), start=1):
    if row.get(STATUS_COL) in {"ok", "empty"}:
        print(f"[{position}/{num_rows}] Skipping completed row {index}")
        continue

    value = row.get(TEXT_COL)
    text = "" if pd.isna(value) else str(value)

    if not text.strip():
        df.at[index, OUTPUT_COL] = ""
        df.at[index, ENTITIES_COL] = "{}"
        df.at[index, REJECTED_COL] = "{}"
        df.at[index, STATUS_COL] = "empty"
        df.at[index, ERROR_COL] = ""
        print(f"[{position}/{num_rows}] Empty row {index}")
        continue

    print(f"[{position}/{num_rows}] Processing row {index}")

    try:
        raw_output = run_ollama_chat(text)
        categories = parse_categories(raw_output)
        found, missing = split_found_and_missing(text, categories)

        masked_text = apply_mask(text, found)
        num_masked = sum(len(v) for v in found.values())
        num_missing = sum(len(v) for v in missing.values())

        df.at[index, OUTPUT_COL] = masked_text
        df.at[index, ENTITIES_COL] = json.dumps(found, ensure_ascii=False)
        df.at[index, REJECTED_COL] = json.dumps(missing, ensure_ascii=False)
        df.at[index, STATUS_COL] = "ok"
        df.at[index, ERROR_COL] = (
            f"{num_missing} strings reported by model not found in source text."
            if num_missing else ""
        )
        print(f"Masked {num_masked} entities" + (f", {num_missing} not found in text" if num_missing else ""))

    except requests.exceptions.Timeout:
        df.at[index, OUTPUT_COL] = pd.NA
        df.at[index, ENTITIES_COL] = "{}"
        df.at[index, REJECTED_COL] = "{}"
        df.at[index, STATUS_COL] = "request_failed"
        df.at[index, ERROR_COL] = "Request timed out."
        print("Request timed out.")
    except Exception as error:
        df.at[index, OUTPUT_COL] = pd.NA
        df.at[index, ENTITIES_COL] = "{}"
        df.at[index, REJECTED_COL] = "{}"
        df.at[index, STATUS_COL] = "failed"
        df.at[index, ERROR_COL] = str(error)
        print(f"Error: {error}")

    if position % CHECKPOINT_EVERY == 0:
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Checkpoint saved to {OUTPUT_CSV}\n")

df.to_csv(OUTPUT_CSV, index=False)

print("\nProcessing completed.")
print(df[STATUS_COL].value_counts(dropna=False))
print(f"Saved to {OUTPUT_CSV}")

Resuming from SCOTBESS_DATASET_MASKED.csv
Found 1675 rows.

[1/1675] Skipping completed row 0
[2/1675] Skipping completed row 1
[3/1675] Skipping completed row 2
[4/1675] Skipping completed row 3
[5/1675] Skipping completed row 4
[6/1675] Skipping completed row 5
[7/1675] Skipping completed row 6
[8/1675] Skipping completed row 7
[9/1675] Skipping completed row 8
[10/1675] Skipping completed row 9
[11/1675] Skipping completed row 10
[12/1675] Skipping completed row 11
[13/1675] Skipping completed row 12
[14/1675] Skipping completed row 13
[15/1675] Skipping completed row 14
[16/1675] Skipping completed row 15
[17/1675] Skipping completed row 16
[18/1675] Skipping completed row 17
[19/1675] Skipping completed row 18
[20/1675] Skipping completed row 19
[21/1675] Skipping completed row 20
[22/1675] Skipping completed row 21
[23/1675] Skipping completed row 22
[24/1675] Skipping completed row 23
[25/1675] Skipping completed row 24
[26/1675] Skipping completed row 25
[27/1675] Skipping comp

In [ ]:
#the one error row will be masked manually

In [ ]:
df = pd.read_csv("SCOTBESS_DATASET_MASKED.csv", encoding="utf-8")
df.to_excel("SCOTBESS_DATASET_MASKED.xlsx", index=False)